# Interpolation with Cubic Spline

This is from YSSY code.

The interpolation using cubic spline might be useful. So I will extract that part and try to improve it and wrap it into a module.

Other things are just standard data processing procedures so we don't need to worry about them.

## Prerequisites

To use cubic spline interpolation, the data must satisfy the following conditions:

1. **Wind field must be in the form of U/V components** (`u_component`, `v_component`), not wind speed / wind direction. Use `convert_wind_to_uv()` to transform if needed.

2. **Gaps must be small (default ≤ 5 consecutive missing steps)** to ensure reliable local spline fitting. Larger gaps are skipped to avoid introducing unreliable extrapolated values.

3. **Enough context points must exist** on both sides of each gap (default ≥ 4 unique known values) to fit a cubic spline. If insufficient, the gap is skipped.

4. **Timestamps should be uniformly spaced** (e.g., 30-minute intervals) for the spline to produce physically meaningful results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import random
from scipy.interpolate import UnivariateSpline
from typing import Optional, Union

---

## Layer 1: Spline Gap Interpolator

Lowest-level atomic operation: given a time series and the position of a single NaN gap, fit a cubic spline on nearby context points and predict values for the missing timestamps.

In [ ]:
DEFAULT_CONTEXT_POINTS = 6
DEFAULT_SPLINE_ORDER = 3
DEFAULT_SPLINE_SMOOTHING = 1
DEFAULT_MAX_GAP_STEPS = 5


def cubic_spline_interpolate_gap(series: pd.Series,
                                  gap_start: int,
                                  gap_end: int,
                                  context_points: int = DEFAULT_CONTEXT_POINTS,
                                  k: int = DEFAULT_SPLINE_ORDER,
                                  s: float = DEFAULT_SPLINE_SMOOTHING):
    """
    Interpolate a single contiguous NaN gap using a cubic spline fitted
    on known data points immediately before and after the gap.

    Parameters
    ----------
    series : pd.Series
        The original (unfilled) series containing the NaN gap.
    gap_start : int
        iloc index of the first NaN in the gap.
    gap_end : int
        iloc index of the last NaN in the gap.
    context_points : int
        Maximum number of known (non-NaN) points to collect from each side.
    k : int
        Degree of the smoothing spline. k=3 gives cubic.
    s : float
        Smoothing factor. s=0 forces exact interpolation through all context
        points; s>0 allows the spline to smooth over noise.

    Returns
    -------
    result : pd.Series or None
        Interpolated values indexed by the gap timestamps, or None on failure.
    meta : dict
        Metadata: {'success': bool, 'context_count': int, 'reason': str}
    """
    n = len(series)

    x_known = []
    y_known = []

    ctx_before_start = max(0, gap_start - context_points)
    ctx_before_end = gap_start - 1
    if ctx_before_end >= ctx_before_start:
        subset = series.iloc[ctx_before_start : ctx_before_end + 1].dropna()
        if not subset.empty:
            x_known.extend([series.index.get_loc(idx) for idx in subset.index])
            y_known.extend(subset.values.tolist())

    ctx_after_start = gap_end + 1
    ctx_after_end = min(n - 1, gap_end + context_points)
    if ctx_after_end >= ctx_after_start:
        subset = series.iloc[ctx_after_start : ctx_after_end + 1].dropna()
        if not subset.empty:
            x_known.extend([series.index.get_loc(idx) for idx in subset.index])
            y_known.extend(subset.values.tolist())

    if len(x_known) < k + 1:
        return None, {
            "success": False,
            "context_count": len(x_known),
            "reason": f"need >= {k + 1} context points, got {len(x_known)}"
        }

    x_arr = np.array(x_known)
    y_arr = np.array(y_known)
    order = np.argsort(x_arr)
    x_sorted = x_arr[order]
    y_sorted = y_arr[order]
    x_unique, unique_idx = np.unique(x_sorted, return_index=True)
    y_unique = y_sorted[unique_idx]

    if len(x_unique) < k + 1:
        return None, {
            "success": False,
            "context_count": len(x_unique),
            "reason": f"need >= {k + 1} unique x values, got {len(x_unique)}"
        }

    try:
        spline = UnivariateSpline(x_unique, y_unique, k=k, s=s)
        gap_indices = np.arange(gap_start, gap_end + 1)
        interpolated_values = spline(gap_indices)
        result = pd.Series(interpolated_values, index=series.index[gap_indices])
        return result, {"success": True, "context_count": len(x_unique)}
    except Exception as e:
        return None, {
            "success": False,
            "context_count": len(x_unique),
            "reason": str(e)
        }

---

## Layer 2: Univariate Series Processor

Operates on a single weather element's time series. Automatically detects all NaN gaps, dispatches small gaps to Layer 1, and skips large gaps.

In [ ]:
def interpolate_series(series: pd.Series,
                       max_gap_steps: int = DEFAULT_MAX_GAP_STEPS,
                       context_points: int = DEFAULT_CONTEXT_POINTS,
                       k: int = DEFAULT_SPLINE_ORDER,
                       s: float = DEFAULT_SPLINE_SMOOTHING):
    """
    Process a single weather element series: detect all NaN gaps,
    interpolate small gaps via cubic spline, skip large gaps.

    Parameters
    ----------
    series : pd.Series
        Time series for one weather element. Index should be uniformly spaced.
    max_gap_steps : int
        Maximum consecutive NaN steps to attempt interpolation.
    context_points : int
        Context points per side for spline fitting.
    k : int
        Spline degree.
    s : float
        Smoothing factor.

    Returns
    -------
    result : pd.Series
        Series with small NaN gaps filled by cubic spline.
    events : list of dict
        Each dict describes a detected gap and the outcome.
    """
    result = series.copy()
    is_na = series.isna()
    n = len(series)
    events = []
    gap_start = -1

    for i in range(n):
        if is_na.iloc[i] and gap_start == -1:
            gap_start = i
        elif (not is_na.iloc[i] or i == n - 1) and gap_start != -1:
            gap_end = i - 1 if not is_na.iloc[i] else i
            gap_length = gap_end - gap_start + 1

            if gap_length <= max_gap_steps:
                interp_vals, meta = cubic_spline_interpolate_gap(
                    series, gap_start, gap_end, context_points, k, s
                )
                if interp_vals is not None:
                    result.loc[interp_vals.index] = interp_vals.values
                    events.append({
                        "time_start": series.index[gap_start],
                        "time_end": series.index[gap_end],
                        "gap_length": gap_length,
                        "method": f"spline(k={k},s={s},ctx={context_points})",
                        "success": True
                    })
                else:
                    events.append({
                        "time_start": series.index[gap_start],
                        "time_end": series.index[gap_end],
                        "gap_length": gap_length,
                        "method": "spline",
                        "success": False,
                        "reason": meta.get("reason", "unknown")
                    })
            else:
                events.append({
                    "time_start": series.index[gap_start],
                    "time_end": series.index[gap_end],
                    "gap_length": gap_length,
                    "method": "skipped",
                    "success": False,
                    "reason": f"gap length {gap_length} > max {max_gap_steps}"
                })

            gap_start = -1

    return result, events

---

## Utility: Wind Conversion

Conversion between wind speed/direction and U/V components. The U/V form is required before running the spline interpolation pipeline.

In [ ]:
def convert_wind_to_uv(df: pd.DataFrame,
                       speed_col: str = "wind_speed",
                       dir_col: str = "wind_dir"):
    """
    Compute U/V wind components from speed and direction.

    Meteorological convention (wind FROM direction):
        u = -speed * sin(direction_rad)
        v = -speed * cos(direction_rad)

    The original speed and direction columns are dropped.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain `speed_col` and `dir_col`.
    speed_col : str
        Column name for wind speed.
    dir_col : str
        Column name for wind direction (degrees).

    Returns
    -------
    pd.DataFrame
        DataFrame with 'u_component' and 'v_component' added, speed/dir removed.
    """
    df = df.copy()
    if speed_col not in df.columns or dir_col not in df.columns:
        raise KeyError(f"Columns '{speed_col}' or '{dir_col}' not found")

    speed = pd.to_numeric(df[speed_col], errors='coerce')
    direction = pd.to_numeric(df[dir_col], errors='coerce')
    dir_rad = np.deg2rad(direction.astype(float))

    df["u_component"] = -speed * np.sin(dir_rad)
    df["v_component"] = -speed * np.cos(dir_rad)

    nan_mask = speed.isna() | direction.isna()
    df.loc[nan_mask, "u_component"] = np.nan
    df.loc[nan_mask, "v_component"] = np.nan

    df = df.drop(columns=[speed_col, dir_col], errors='ignore')
    return df


def convert_uv_to_wind(df: pd.DataFrame):
    """
    Recalculate wind speed and direction from U/V components.

    Adds 'wind_speed_recalc' and 'wind_dir_recalc' columns.
    Direction is set to NaN when speed ≤ 0.01 kts (calm wind).
    """
    df = df.copy()
    if "u_component" not in df.columns or "v_component" not in df.columns:
        raise KeyError("'u_component' and 'v_component' columns required")

    df["wind_speed_recalc"] = np.sqrt(
        df["u_component"] ** 2 + df["v_component"] ** 2
    )
    dir_rad = np.arctan2(-df["u_component"], -df["v_component"])
    df["wind_dir_recalc"] = (np.rad2deg(dir_rad) + 360) % 360
    df["wind_dir_recalc"] = df["wind_dir_recalc"].where(
        df["wind_speed_recalc"].notna() & (df["wind_speed_recalc"] > 0.01),
        np.nan
    )
    return df

---

## Layer 3: Multivariate DataFrame Coordinator

Orchestrates the full pipeline across multiple weather elements. Handles dependency ordering (U/V before derived wind), completeness recalculation, and summary reporting.

In [ ]:
SCALAR_ELEMENTS_DEFAULT = ["air_temp", "dew_point", "msl_pressure"]
UV_ELEMENTS = ["u_component", "v_component"]


def interpolate_dataframe(df: pd.DataFrame,
                          elements: Optional[list] = None,
                          max_gap_steps: int = DEFAULT_MAX_GAP_STEPS,
                          context_points: int = DEFAULT_CONTEXT_POINTS,
                          k: int = DEFAULT_SPLINE_ORDER,
                          s: float = DEFAULT_SPLINE_SMOOTHING):
    """
    Main entry point: interpolate multiple weather elements in a DataFrame.

    Processing order:
    1. U/V components (if present) — these are the source of wind information.
    2. Scalar elements (air_temp, dew_point, msl_pressure).
    3. Recalculate wind_speed_recalc / wind_dir_recalc from (now-interpolated) U/V.
    4. Recalculate data_completeness.

    Parameters
    ----------
    df : pd.DataFrame
        Data with DatetimeIndex. Must contain at least one of the specified elements.
    elements : list of str, optional
        Columns to interpolate. Defaults to all scalar + UV elements present.
    max_gap_steps : int
        Maximum consecutive NaN steps to attempt interpolation.
    context_points : int
        Context points per side for spline fitting.
    k : int
        Spline degree.
    s : float
        Smoothing factor.

    Returns
    -------
    df_processed : pd.DataFrame
        DataFrame with small gaps filled and derived columns added.
    report : dict
        Summary of gaps detected and filled per element.
    """
    df_processed = df.copy()

    if elements is None:
        elements = [el for el in SCALAR_ELEMENTS_DEFAULT + UV_ELEMENTS
                     if el in df_processed.columns]

    has_uv = all(el in elements for el in UV_ELEMENTS)
    ordered_elements = []
    if has_uv:
        ordered_elements.extend(UV_ELEMENTS)
    for el in elements:
        if el not in ordered_elements and el in df_processed.columns:
            ordered_elements.append(el)

    all_events = {}
    report = {
        "elements": {},
        "total_gaps_found": 0,
        "total_gaps_filled": 0,
        "total_gaps_skipped": 0,
        "params": {
            "max_gap_steps": max_gap_steps,
            "context_points": context_points,
            "spline_order": k,
            "smoothing": s
        }
    }

    for element in ordered_elements:
        if element not in df_processed.columns:
            continue

        if not pd.api.types.is_numeric_dtype(df_processed[element]):
            df_processed[element] = pd.to_numeric(
                df_processed[element], errors='coerce'
            )

        interp_series, events = interpolate_series(
            df_processed[element], max_gap_steps, context_points, k, s
        )
        df_processed[element] = interp_series

        filled = sum(1 for e in events if e["success"])
        skipped = len(events) - filled
        report["elements"][element] = {
            "gaps_found": len(events),
            "gaps_filled": filled,
            "gaps_skipped": skipped
        }
        report["total_gaps_found"] += len(events)
        report["total_gaps_filled"] += filled
        report["total_gaps_skipped"] += skipped
        all_events[element] = events

    if has_uv:
        df_processed = convert_uv_to_wind(df_processed)

    completeness_cols = ["air_temp", "dew_point", "msl_pressure"]
    if "wind_speed_recalc" in df_processed.columns:
        completeness_cols.append("wind_speed_recalc")
        completeness_cols.append("wind_dir_recalc")
    existing = [c for c in completeness_cols if c in df_processed.columns]
    if existing:
        df_processed["data_completeness"] = (
            df_processed[existing].notna().all(axis=1).astype(int)
        )

    report["all_events"] = all_events
    return df_processed, report


def print_report(report: dict):
    """Pretty-print the interpolation summary report."""
    params = report["params"]
    print("=" * 60)
    print("CUBIC SPLINE INTERPOLATION REPORT")
    print("=" * 60)
    print(f"Spline: k={params['spline_order']}, s={params['smoothing']}, "
          f"context={params['context_points']}, max_gap={params['max_gap_steps']}")
    print(f"Total gaps found:  {report['total_gaps_found']}")
    print(f"Total gaps filled: {report['total_gaps_filled']}")
    print(f"Total gaps skipped:{report['total_gaps_skipped']}")
    print("-" * 60)
    print(f"{'Element':<20} {'Found':>6} {'Filled':>6} {'Skipped':>7}")
    print("-" * 60)
    for el, stats in report["elements"].items():
        print(f"{el:<20} {stats['gaps_found']:>6} {stats['gaps_filled']:>6} "
              f"{stats['gaps_skipped']:>7}")
    print("=" * 60)

---

## Validation: Synthetic Gap Test

Carve an artificial gap out of known data, interpolate, and compare against the hidden true values. This measures how well the spline recovers the underlying signal.

In [ ]:
def validate_synthetic_gap(series: pd.Series,
                           gap_start: int,
                           gap_length: int,
                           context_points: int = DEFAULT_CONTEXT_POINTS,
                           k: int = DEFAULT_SPLINE_ORDER,
                           s: float = DEFAULT_SPLINE_SMOOTHING):
    """
    Test interpolation accuracy by removing a chunk of known data,
    interpolating it, and comparing against the true values.

    Parameters
    ----------
    series : pd.Series
        A clean (NaN-free) segment of data.
    gap_start : int
        iloc index where the artificial gap begins.
    gap_length : int
        Number of steps to remove.
    context_points, k, s :
        Passed through to cubic_spline_interpolate_gap.

    Returns
    -------
    result : dict or None
        {'true': pd.Series, 'predicted': pd.Series, 'rmse': float, 'mae': float}
        or None if interpolation failed.
    """
    true_slice = series.iloc[gap_start : gap_start + gap_length]
    if true_slice.isna().any():
        raise ValueError("Gap region contains NaN in original data")

    test_series = series.copy()
    test_series.iloc[gap_start : gap_start + gap_length] = np.nan

    gap_end = gap_start + gap_length - 1
    interp_vals, meta = cubic_spline_interpolate_gap(
        test_series, gap_start, gap_end, context_points, k, s
    )

    if interp_vals is None:
        return None

    true_vals = true_slice.values
    pred_vals = interp_vals.values
    rmse = np.sqrt(np.mean((true_vals - pred_vals) ** 2))
    mae = np.mean(np.abs(true_vals - pred_vals))

    return {"true": true_slice, "predicted": interp_vals, "rmse": rmse, "mae": mae}


def run_validation_sweep(series: pd.Series,
                         gap_lengths: list = None,
                         num_samples: int = 20,
                         **spline_params):
    """
    Run multiple synthetic gap tests across random positions.

    Returns a DataFrame of results (one row per test gap).
    """
    if gap_lengths is None:
        gap_lengths = [1, 2, 3, 4, 5]

    clean = series.dropna()
    results = []

    for gl in gap_lengths:
        max_start = len(clean) - gl - spline_params.get("context_points", 6) - 1
        min_start = spline_params.get("context_points", 6)

        if max_start <= min_start:
            continue

        positions = np.linspace(min_start, max_start, num_samples, dtype=int)
        for pos in positions:
            result = validate_synthetic_gap(clean, pos, gl, **spline_params)
            if result is not None:
                result["gap_length"] = gl
                result["position"] = pos
                results.append(result)

    return pd.DataFrame(results)

---

## Visualization: Interpolation Sample Plot

For qualitative inspection, plot the interpolated section in context with surrounding data. The interpolated segment is highlighted with a thick line.

In [ ]:
PLOT_COLORS = {
    "air_temp": "#e74c3c",
    "dew_point": "#3498db",
    "wind_speed_recalc": "#2c3e50",
    "wind_dir_recalc": "#7f8c8d",
    "u_component": "#27ae60",
    "v_component": "#e67e22",
    "msl_pressure": "#9b59b6",
}


def plot_interpolation_sample(df: pd.DataFrame,
                               element_name: str,
                               gap_event: dict,
                               context_hours: float = 12.0,
                               figsize: tuple = (16, 8)):
    """
    Plot an interpolated gap with surrounding context for visual inspection.

    Parameters
    ----------
    df : pd.DataFrame
        Fully processed DataFrame (post-interpolation), with DatetimeIndex.
    element_name : str
        Column name of the element that was interpolated.
    gap_event : dict
        One event dict from the events log, must have 'time_start' and 'time_end'.
    context_hours : float
        Hours of data to show before and after the gap.
    """
    gap_start = gap_event["time_start"]
    gap_end = gap_event["time_end"]
    ctx_delta = pd.Timedelta(hours=context_hours)

    plot_start = gap_start - ctx_delta
    plot_end = gap_end + ctx_delta
    slice_data = df.loc[plot_start:plot_end]

    if slice_data.empty:
        print("Empty context slice. Skipping plot.")
        return

    fig, ax = plt.subplots(figsize=figsize)

    el_label = element_name.replace("_", " ").title()
    color = PLOT_COLORS.get(element_name, "gray")

    before = slice_data.loc[slice_data.index < gap_start]
    during = slice_data.loc[gap_start:gap_end]
    after = slice_data.loc[slice_data.index > gap_end]

    if not before.empty:
        ax.plot(before.index, before[element_name],
                color=color, linewidth=1.5, linestyle="-", alpha=0.7)
    if not during.empty:
        ax.plot(during.index, during[element_name],
                color=color, linewidth=3.0, linestyle="-",
                label=f"{el_label} (interpolated)")
    if not after.empty:
        ax.plot(after.index, after[element_name],
                color=color, linewidth=1.5, linestyle="-", alpha=0.7)

    ax.axvspan(gap_start, gap_end, alpha=0.1, color=color,
               label=f"Gap: {gap_event['gap_length']} steps")

    method = gap_event.get("method", "unknown")
    ax.set_title(
        f"Spline Interpolation — {el_label}\n"
        f"Gap: {gap_start} → {gap_end} ({gap_event['gap_length']} steps) | "
        f"Method: {method}",
        fontsize=13
    )
    ax.set_xlabel("Timestamp")
    ax.set_ylabel(el_label)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="best")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

---

## Usage Example

End-to-end demonstration: load data, run the pipeline, inspect the report, and plot a sample.

In [ ]:
# Step 1: Load data (assumes .txt file with timestamp column)
DATA_PATH = "../scripts/YSSY-code/UVComponentData/YSSY.txt"

try:
    raw = pd.read_csv(DATA_PATH, parse_dates=["timestamp"], na_values=["NaN"])
    raw = raw.set_index("timestamp")
    print(f"Loaded: {raw.shape}")
except FileNotFoundError:
    # Fallback: create synthetic data for demonstration
    print("Data file not found. Generating synthetic demo data.")
    np.random.seed(42)
    n = 1000
    times = pd.date_range("2020-01-01", periods=n, freq="30min")
    trend = np.sin(np.linspace(0, 8 * np.pi, n)) * 5 + 25
    noise = np.random.normal(0, 0.5, n)

    raw = pd.DataFrame({
        "air_temp": trend + noise,
        "dew_point": trend * 0.6 + noise * 0.7 + 10,
        "msl_pressure": 1013 + np.sin(np.linspace(0, 4 * np.pi, n)) * 5 + noise * 0.3,
        "u_component": np.sin(np.linspace(0, 10 * np.pi, n)) * 8 + noise * 0.8,
        "v_component": np.cos(np.linspace(0, 10 * np.pi, n)) * 6 + noise * 0.8,
    }, index=times)

    # Artificially introduce small gaps
    for col, gaps in [
        ("air_temp", [(100, 2), (350, 4), (700, 1)]),
        ("u_component", [(200, 3), (500, 5)]),
        ("v_component", [(200, 3), (500, 5)]),
        ("msl_pressure", [(400, 2)]),
    ]:
        for start, length in gaps:
            raw.iloc[start:start+length, raw.columns.get_loc(col)] = np.nan

    print(f"Synthetic data created: {raw.shape}")

print(f"NaN count before interpolation:\n{raw.isna().sum()}")

In [ ]:
# Step 2: Run the interpolation pipeline
df_interpolated, report = interpolate_dataframe(
    raw,
    max_gap_steps=5,
    context_points=6,
    k=3,
    s=1
)

print_report(report)
print(f"\nNaN count after interpolation:\n{df_interpolated.isna().sum()}")

In [ ]:
# Step 3: Plot a sample interpolated gap for visual inspection
all_events = report.get("all_events", {})
filled_events = []
for el, events in all_events.items():
    for ev in events:
        if ev["success"]:
            filled_events.append((el, ev))

if filled_events:
    el, ev = random.choice(filled_events)
    plot_interpolation_sample(df_interpolated, el, ev, context_hours=6)
else:
    print("No successful interpolation events to plot.")

In [ ]:
# Step 4 (optional): Validate accuracy with synthetic gap tests
# Pick a clean segment and carve artificial gaps to measure RMSE
clean_segment = raw["air_temp"].dropna()
if len(clean_segment) > 50:
    val_results = run_validation_sweep(
        clean_segment,
        gap_lengths=[1, 2, 3, 4, 5],
        num_samples=10,
        context_points=6, k=3, s=1
    )
    if not val_results.empty:
        summary = val_results.groupby("gap_length")[["rmse", "mae"]].agg(["mean", "std"])
        print("Validation RMSE/MAE by gap length:")
        print(summary.to_string())